In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm
from utils import *
from gudhi import AlphaComplex, RipsComplex, SimplexTree
from gudhi.representations import BettiCurve
import chromatic_tda as chro

In [2]:
data_folder = Path('data')
xt = 'files/xt.csv'
xt_df = pd.read_csv(xt)

### Non-topological features

In [3]:
rows = []
len_xt = len(xt_df)
for _, row in tqdm(xt_df.iterrows(),total=len_xt, desc='Progress'):
    x = row['x']
    t = row['t']
    label = row['label']
    filename = f'ID-{x}_time-{t}_From2ParamSweep_Data.csv'
    file_path = data_folder / filename
    if file_path.exists():
        df = pd.read_csv(file_path)
        M = df.loc[df['celltypes'] == 'Macrophage', ['points_x', 'points_y']].to_numpy()
        T = df.loc[df['celltypes'] == 'Tumour', ['points_x', 'points_y']].to_numpy()
        N = df.loc[df['celltypes'] == 'Necrotic', ['points_x', 'points_y']].to_numpy()
        V = df.loc[df['celltypes'] == 'Vessel', ['points_x', 'points_y']].to_numpy()
        Mcount = len(M)
        Tcount = len(T)
        Ncount = len(N)
        MVdist = avg_dist(M, V)
        TVdist = avg_dist(T, V)
        NVdist = avg_dist(N, V)
        rows.append([x,t,label,
                    Mcount,Tcount,Ncount,
                    MVdist,TVdist,NVdist])

Progress: 100%|███████████████████████████████████████████████████████████████████████| 919/919 [00:59<00:00, 15.53it/s]


In [4]:
columns = ['x', 't', 'label','Mcount', 'Tcount', 'Ncount', 'MVdist', 'TVdist', 'NVdist']
results = pd.DataFrame(rows, columns=columns)
results = results.sort_values(by=['x', 't'], ascending=[True, True])
results.to_csv('files/xt_nontop.csv',index=False)

### L-fuzzy Betti curves for Alpha complexes

In [5]:
homology_coeff_field = 2
predefined_grid = np.arange(0, 10, 0.2)

In [6]:
rows = []
len_xt = len(xt_df)
for _, row in tqdm(xt_df.iterrows(), total=len_xt, desc='Progress'):
    x = row['x']
    t = row['t']
    label = row['label']
    filename = f'ID-{x}_time-{t}_From2ParamSweep_Data.csv'
    file_path = data_folder / filename
    if file_path.exists():
        df = pd.read_csv(file_path)
        celltypes = {
            'M': df.loc[df['celltypes'] == 'Macrophage',
                        ['points_x', 'points_y']].to_numpy(),
            'T': df.loc[df['celltypes'] == 'Tumour',
                        ['points_x', 'points_y']].to_numpy(),
            'V': df.loc[df['celltypes'] == 'Vessel',
                        ['points_x', 'points_y']].to_numpy(),
        }
        curves = []
        for pair in [['M','T'],['T','V'],['M','V']]:
            P1 = celltypes[pair[0]]
            l1 = len(P1)
            P2 = celltypes[pair[1]]
            l2 = len(P2)
            data = np.vstack([P1, P2])
            fuzzy_values = ['x'] * len(P1) + ['y'] * len(P2)
            comp = AlphaComplex(points=data)
            st = comp.create_simplex_tree(output_squared_values=False)
            fbcurves = FuzzyBettiCurve(simplex_tree=st, fuzzy_values=fuzzy_values, predefined_grid=predefined_grid)
            curves.extend([fbcurves[0].flatten(),fbcurves[1].flatten()])
        rows.append(np.concatenate([[x, t, label]] + curves))

Progress: 100%|███████████████████████████████████████████████████████████████████████| 919/919 [23:35<00:00,  1.54s/it]


In [7]:
columns = ['x', 't','label'] + [f'{pair}{dim}_{val}_{i}' 
                        for pair in ['MT', 'TV', 'MV'] 
                        for dim in ['0', '1'] 
                        for val in ['xandy','x','y','xory']
                        for i in range(len(predefined_grid))]
results = pd.DataFrame(rows, columns=columns,dtype=int)
results = results.sort_values(by=['x', 't'], ascending=[True, True])
results.to_csv('files/xt_alpha.csv',index=False)

### L-fuzzy Betti curves for Chromatic Alpha complexes

In [8]:
rows = []
len_xt = len(xt_df)
for _, row in tqdm(xt_df.iterrows(), total=len_xt, desc='Progress'):
    x = row['x']
    t = row['t']
    label = row['label']
    filename = f'ID-{x}_time-{t}_From2ParamSweep_Data.csv'
    file_path = data_folder / filename
    if file_path.exists():
        df = pd.read_csv(file_path)
        celltypes = {
            'M': df.loc[df['celltypes'] == 'Macrophage',
                        ['points_x', 'points_y']].to_numpy(),
            'T': df.loc[df['celltypes'] == 'Tumour',
                        ['points_x', 'points_y']].to_numpy(),
            'V': df.loc[df['celltypes'] == 'Vessel',
                        ['points_x', 'points_y']].to_numpy(),
        }
        curves = []
        for pair in [['M','T'],['T','V'],['M','V']]:
            P1 = celltypes[pair[0]]
            l1 = len(P1)
            P2 = celltypes[pair[1]]
            l2 = len(P2)
            data = np.vstack([P1, P2])
            fuzzy_values = ['x'] * len(P1) + ['y'] * len(P2)
            try:
                comp = chro.ChromaticAlphaComplex(points=data,labels=fuzzy_values)
            except:
                comp = chro.ChromaticAlphaComplex(points=data,labels=fuzzy_values,point_perturbation=1e-10)
            st = comp.get_simplicial_complex(sub_complex='mono-chromatic')
            simplices = st.weight_function()
            crst = SimplexTree()
            for simplex, filt in sorted(simplices.items(), key=lambda x: len(x[0])):
                crst.insert(simplex, filtration=float(filt))
            fbcurves = FuzzyBettiCurve(simplex_tree=crst, fuzzy_values=fuzzy_values, predefined_grid=predefined_grid)
            curves.extend([fbcurves[0].flatten(),fbcurves[1].flatten()])
        rows.append(np.concatenate([[x, t, label]] + curves))

Progress: 100%|█████████████████████████████████████████████████████████████████████| 919/919 [1:55:25<00:00,  7.54s/it]


In [9]:
columns = ['x', 't','label'] + [f'{pair}{dim}_{val}_{i}' 
                        for pair in ['MT', 'TV', 'MV'] 
                        for dim in ['0', '1'] 
                        for val in ['xandy','x','y','xory']
                        for i in range(len(predefined_grid))]
results = pd.DataFrame(rows, columns=columns,dtype=int)
results = results.sort_values(by=['x', 't'], ascending=[True, True])
results.to_csv('files/xt_calpha.csv',index=False)

### Betti curves of Six-pack

In [10]:
rows = []
len_xt = len(xt_df)
for _, row in tqdm(xt_df.iterrows(), total=len_xt, desc='Progress'):
    x = row['x']
    t = row['t']
    label = row['label']
    filename = f'ID-{x}_time-{t}_From2ParamSweep_Data.csv'
    file_path = data_folder / filename
    if file_path.exists():
        df = pd.read_csv(file_path)
        celltypes = {
            'M': df.loc[df['celltypes'] == 'Macrophage',
                        ['points_x', 'points_y']].to_numpy(),
            'T': df.loc[df['celltypes'] == 'Tumour',
                        ['points_x', 'points_y']].to_numpy(),
            'V': df.loc[df['celltypes'] == 'Vessel',
                        ['points_x', 'points_y']].to_numpy(),
        }
        curves = []
        for pair in [['M','T'],['T','V'],['M','V']]:
            P1 = celltypes[pair[0]]
            l1 = len(P1)
            P2 = celltypes[pair[1]]
            l2 = len(P2)
            data = np.vstack([P1, P2])
            fuzzy_values = ['x'] * len(P1) + ['y'] * len(P2)
            try:
                comp = chro.ChromaticAlphaComplex(points=data,labels=fuzzy_values)
            except:
                comp = chro.ChromaticAlphaComplex(points=data,labels=fuzzy_values,point_perturbation=1e-10)
            st = comp.get_simplicial_complex(sub_complex='mono-chromatic')
            st.compute_persistence()
            sixpack = st.bars_six_pack()
            for dim in [0,1]:
                for key in ['sub_complex','complex','relative','image','kernel','cokernel']:
                    if dim in sixpack[key].keys():
                        curves.extend(betti_curve_from_barcode(sixpack[key][dim], predefined_grid))
                    else:
                        curves.extend([0]*len(predefined_grid))
        rows.append([x, t, label] + curves)

Progress: 100%|█████████████████████████████████████████████████████████████████████| 919/919 [1:26:25<00:00,  5.64s/it]


In [12]:
columns = ['x', 't','label'] + [f'{pair}{dim}_{val}_{i}' 
                        for pair in ['MT', 'TV', 'MV'] 
                        for dim in ['0', '1'] 
                        for val in ['sub_complex','complex','relative','image','kernel','cokernel']
                        for i in range(len(predefined_grid))]
results = pd.DataFrame(rows, columns=columns,dtype=int)
results = results.sort_values(by=['x', 't'], ascending=[True, True])
results.to_csv('files/xt_sixpack.csv',index=False)